# CS-781 Phase 2 Legal NER — V2 Encoder + LORMA Character-Boundary Adaptation

**Experiment:** EXP-2V2-LORMA-CharacterBoundary  
**Backbone:** the second Phase 1 architecture, `V2LegalLM` encoder from the supplied encoder-decoder notebook  
**Adaptation:** low-rank model adaptation (LORMA bottleneck residual adapters) in encoder attention and feed-forward projections  
**Prediction:** exact character-level BIO spans, with validation-only confidence calibration  
**Training:** up to 6 epochs, best checkpoint selected by held-out strict span Macro-F1

This notebook does not submit anything. It only creates and validates a `prediction.json` and ZIP artifact for you to submit manually.

**Required input:** a trained Phase 1 V2 checkpoint saved as `v2_model.pt` (the checkpoint produced by the supplied second-architecture notebook), the matching `bpe_en_25k.json` tokenizer, and the Phase 2 data. Attach the checkpoint/tokenizer/data as Kaggle inputs or set the explicit paths below.

The model code preserves the V2 parameter names while adding padding-mask support to the encoder. This is important because otherwise padded tokens can contaminate representations in variable-length batches.

## 1. Runtime, reproducibility, and paths

In [ ]:
import os, json, re, math, time, random, copy, zipfile, hashlib, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF") or HF_TOKEN
    except Exception:
        pass
except Exception:
    pass

# Set these if automatic discovery finds no unique checkpoint/tokenizer.
V2_CHECKPOINT_PATH = ""  # e.g. /kaggle/input/my-v2-checkpoint/v2_model.pt
BPE_PATH = ""            # e.g. /kaggle/input/my-tokenizer/bpe_en_25k.json

PHASE2_ROOT_DIR = "/kaggle/working/cs781_phase2_v2_lorma"
CHECKPOINT_DIR = os.path.join(PHASE2_ROOT_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, "artifacts")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TOKEN_LENGTH = 512
WINDOW_OVERLAP = 256
EPOCHS = 6
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_DROPOUT = 0.05
LORMA_LR = 1.5e-4
CHAR_HEAD_LR = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.08
MAX_GRAD_NORM = 1.0
PATIENCE = 3
USE_AMP = torch.cuda.is_available()

print("Experiment output:", PHASE2_ROOT_DIR)
print("Epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| grad accumulation:", GRAD_ACCUM_STEPS)

## 2. Load the exact BPE-25k tokenizer

In [ ]:
BPE_FILENAME = "bpe_en_25k.json"
if not BPE_PATH:
    candidates = []
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                if BPE_FILENAME in files:
                    candidates.append(os.path.join(root, BPE_FILENAME))
    candidates = sorted(set(candidates))
    if len(candidates) == 1:
        BPE_PATH = candidates[0]
    elif len(candidates) > 1:
        preferred = [p for p in candidates if "bpe_en_25k.json" in p]
        if len(preferred) == 1:
            BPE_PATH = preferred[0]
        else:
            raise RuntimeError("Multiple tokenizer files found. Set BPE_PATH explicitly:\n" + "\n".join(candidates))
if not BPE_PATH or not os.path.isfile(BPE_PATH):
    raise FileNotFoundError("Could not find bpe_en_25k.json. Attach the matching tokenizer as a Kaggle input or set BPE_PATH.")

with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)
bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
assert int(bpe_data["vocab_size"]) == 25000 and len(bpe_itos) == 25000

bpe_stoi = {token: idx for idx, token in enumerate(bpe_itos)}
merge_to_id = {}
for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge
    merge_to_id[(bpe_stoi[left_token], bpe_stoi[right_token])] = 261 + merge_index

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos, self.stoi = itos, stoi
        self.merge_to_id, self.merges = merge_to_id, merges
        self.vocab_size = len(itos)
        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        sequence = [4 + byte for byte in word.encode("utf-8")]
        sequence.append(self.word_boundary_id)
        while len(sequence) > 1:
            best_position, best_token_id = None, None
            for i in range(len(sequence) - 1):
                new_token_id = self.merge_to_id.get((sequence[i], sequence[i + 1]))
                if new_token_id is not None and (best_token_id is None or new_token_id < best_token_id):
                    best_position, best_token_id = i, new_token_id
            if best_position is None:
                break
            i = best_position
            sequence = sequence[:i] + [best_token_id] + sequence[i + 2:]
        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_token_id] if add_bos else []
        for word in text.split():
            ids.extend(self._encode_word(word))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def _token_to_bytes(self, token_id):
        token = self.itos[int(token_id)]
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])
        if token == "<wb>":
            return b" "
        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""
        if token.startswith("<bpe_"):
            merge_index = int(token_id) - 261
            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]
                return self._token_to_bytes(self.stoi[left_token]) + self._token_to_bytes(self.stoi[right_token])
        return b""

    def decode(self, token_ids):
        output = bytearray()
        for token_id in token_ids:
            if 0 <= int(token_id) < self.vocab_size:
                output.extend(self._token_to_bytes(int(token_id)))
        return bytes(output).decode("utf-8", errors="replace")

tokenizer = ByteLevelBPE25k(bpe_itos, bpe_stoi, merge_to_id, bpe_merges)
assert tokenizer.vocab_size == 25000
print("Tokenizer loaded:", BPE_PATH)
print("Tokenizer round trip:", tokenizer.decode(tokenizer.encode("Supreme Court of India")))

## 3. Resolve Phase 2 data files

In [ ]:
from huggingface_hub import hf_hub_download

PHASE2_HF_PATHS = {
    "fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
    "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
    "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json",
    "labels.txt": "Phase_2_LNER/Train/labels.txt",
}
PHASE2_FILES = {}
for filename, hub_filename in PHASE2_HF_PATHS.items():
    local_matches = []
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                if filename in files:
                    local_matches.append(os.path.join(root, filename))
    local_matches = sorted(set(local_matches))
    if len(local_matches) == 1:
        PHASE2_FILES[filename] = local_matches[0]
    elif len(local_matches) > 1:
        # Prefer a path explicitly under Phase_2_LNER if there is one unique match.
        preferred = [p for p in local_matches if "Phase_2_LNER" in p]
        if len(preferred) == 1:
            PHASE2_FILES[filename] = preferred[0]
        else:
            raise RuntimeError(f"Multiple local copies of {filename}; resolve explicitly: {local_matches}")
    else:
        PHASE2_FILES[filename] = hf_hub_download(
            repo_id=HF_DATASET_REPO, filename=hub_filename,
            repo_type="dataset", token=HF_TOKEN or None
        )
    print(filename, "->", PHASE2_FILES[filename])

## 4. Load and validate annotations; split by document

In [ ]:
from collections import Counter

EXPECTED_PHASE2_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}


def _load_json_file(filename):
    with open(PHASE2_FILES[filename], "r", encoding="utf-8") as handle:
        return json.load(handle)


def _normalise_text_map(raw_data, filename):
    # The Phase 2 specification describes text files as document-ID -> full text mappings.
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    result = {}
    for document_id, value in raw_data.items():
        if isinstance(value, str):
            text_value = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            # Supports a wrapped record if the published file uses that variant.
            text_value = value["text"]
        else:
            raise TypeError(
                f"Unexpected text record for document {document_id!r} in {filename}: {type(value).__name__}."
            )
        result[str(document_id)] = text_value
    return result


def _first_present(record, keys, field_name, document_id):
    for key in keys:
        if key in record:
            return record[key]
    raise KeyError(
        f"Missing {field_name!r} field for document {document_id!r}. Expected one of {keys}; got keys {list(record)}."
    )


def _normalise_entities_for_document(raw_entities, document_id, filename):
    # Supports the published [start, end, label, optional_text] format and dictionary variants.
    if raw_entities is None:
        return []
    if isinstance(raw_entities, dict):
        if "entities" in raw_entities:
            raw_entities = raw_entities["entities"]
        elif any(k in raw_entities for k in ("start", "start_char", "start_offset")):
            raw_entities = [raw_entities]
        else:
            raise TypeError(
                f"Unexpected annotation object for document {document_id!r} in {filename}: {list(raw_entities)[:12]}"
            )
    if not isinstance(raw_entities, (list, tuple)):
        raise TypeError(
            f"Annotations for document {document_id!r} in {filename} should be a list; got {type(raw_entities).__name__}."
        )

    entities = []
    for entity_index, entity in enumerate(raw_entities):
        if isinstance(entity, dict):
            start = _first_present(entity, ("start", "start_char", "start_offset", "begin"), "start", document_id)
            end = _first_present(entity, ("end", "end_char", "end_offset", "stop"), "end", document_id)
            label = _first_present(entity, ("label", "entity_label", "type", "entity_type"), "label", document_id)
            entity_text = entity.get("text", entity.get("entity_text", entity.get("surface")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
            entity_text = entity[3] if len(entity) >= 4 else None
        else:
            raise TypeError(
                f"Unexpected entity at index {entity_index} for document {document_id!r} in {filename}: {entity!r}"
            )

        try:
            start = int(start)
            end = int(end)
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"Non-integer span boundaries for document {document_id!r}, entity {entity_index}: {entity!r}"
            ) from exc
        if not isinstance(label, str):
            raise TypeError(f"Entity label must be a string, got {type(label).__name__}: {entity!r}")
        entities.append({"start": start, "end": end, "label": label, "text": entity_text})
    return entities


def _normalise_gold_map(raw_data, filename):
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    return {
        str(document_id): _normalise_entities_for_document(entities, str(document_id), filename)
        for document_id, entities in raw_data.items()
    }


with open(PHASE2_FILES["labels.txt"], "r", encoding="utf-8") as handle:
    labels_lines = [line.strip() for line in handle if line.strip() and not line.strip().startswith("#")]

# Accept either one label per line or a label followed by a human-readable description.
LABELS = []
for line in labels_lines:
    first_token = line.split()[0]
    if first_token in EXPECTED_PHASE2_LABELS:
        candidate = first_token
    elif line in EXPECTED_PHASE2_LABELS:
        candidate = line
    else:
        continue
    if candidate not in LABELS:
        LABELS.append(candidate)

if set(LABELS) != EXPECTED_PHASE2_LABELS:
    raise ValueError(
        "labels.txt did not resolve to the expected 12 Phase 2 labels. "
        f"Found={LABELS}; missing={sorted(EXPECTED_PHASE2_LABELS - set(LABELS))}; "
        f"unexpected={sorted(set(LABELS) - EXPECTED_PHASE2_LABELS)}"
    )

FOLD1_TEXTS = _normalise_text_map(_load_json_file("fold1-text.json"), "fold1-text.json")
FOLD1_GOLD = _normalise_gold_map(_load_json_file("fold1-gold.json"), "fold1-gold.json")
FOLD2_TEXTS = _normalise_text_map(_load_json_file("fold2-text.json"), "fold2-text.json")
FOLD2_GOLD = _normalise_gold_map(_load_json_file("fold2-gold.json"), "fold2-gold.json")
TEST_TEXTS = _normalise_text_map(_load_json_file("fold3-text.json"), "fold3-text.json")

for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    missing_gold = sorted(set(texts) - set(gold))
    orphan_gold = sorted(set(gold) - set(texts))
    if missing_gold or orphan_gold:
        raise ValueError(
            f"{fold_name} text/gold document IDs do not match. "
            f"Missing gold for {len(missing_gold)} IDs; gold without text for {len(orphan_gold)} IDs. "
            f"First missing={missing_gold[:5]}, first orphan={orphan_gold[:5]}"
        )

TEXTS_BY_ID = {}
ANNOTATIONS_BY_ID = {}
for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    overlap = set(TEXTS_BY_ID) & set(texts)
    if overlap:
        raise ValueError(
            f"Document IDs occur in both training folds ({len(overlap)} overlaps). "
            f"Example IDs: {sorted(overlap)[:10]}"
        )
    TEXTS_BY_ID.update(texts)
    ANNOTATIONS_BY_ID.update(gold)

TEST_OVERLAP = set(TEXTS_BY_ID) & set(TEST_TEXTS)
if TEST_OVERLAP:
    raise ValueError(
        f"Training and test document IDs overlap ({len(TEST_OVERLAP)} IDs). Example: {sorted(TEST_OVERLAP)[:10]}"
    )

# Strictly validate offsets and labels before tokenization; do not silently repair annotations.
ANNOTATION_COUNTS = Counter()
INVALID_SPANS = []
ENTITY_TEXT_MISMATCHES = []
for document_id, entities in ANNOTATIONS_BY_ID.items():
    source_text = TEXTS_BY_ID[document_id]
    for entity_index, entity in enumerate(entities):
        start, end, label = entity["start"], entity["end"], entity["label"]
        ANNOTATION_COUNTS[label] += 1
        if label not in EXPECTED_PHASE2_LABELS:
            INVALID_SPANS.append((document_id, entity_index, "unknown_label", label))
        if not (0 <= start < end <= len(source_text)):
            INVALID_SPANS.append((document_id, entity_index, "invalid_offsets", start, end, len(source_text)))
        elif entity["text"] is not None and source_text[start:end] != entity["text"]:
            ENTITY_TEXT_MISMATCHES.append((document_id, entity_index, entity["text"], source_text[start:end]))

if INVALID_SPANS:
    raise ValueError(
        f"Found {len(INVALID_SPANS)} invalid annotations. Examples: {INVALID_SPANS[:10]}"
    )

print("Official labels:", LABELS)
print(f"Fold 1: {len(FOLD1_TEXTS):,} documents, {sum(map(len, FOLD1_GOLD.values())):,} entities")
print(f"Fold 2: {len(FOLD2_TEXTS):,} documents, {sum(map(len, FOLD2_GOLD.values())):,} entities")
print(f"Combined training pool: {len(TEXTS_BY_ID):,} documents")
print(f"Fold 3 test documents: {len(TEST_TEXTS):,}")
print("Entity counts:")
for label in LABELS:
    print(f"  {label:10s} {ANNOTATION_COUNTS[label]:,}")
print("Optional entity_text mismatches (reported only, offsets preserved):", len(ENTITY_TEXT_MISMATCHES))
if ENTITY_TEXT_MISMATCHES:
    print("Mismatch examples:", ENTITY_TEXT_MISMATCHES[:3])

# Deterministic document-level holdout; keep each document entirely in one split.
all_document_ids = list(TEXTS_BY_ID)
random.Random(SEED).shuffle(all_document_ids)
if len(all_document_ids) < 2:
    raise ValueError("At least two labelled documents are required to create train/validation splits.")
VALIDATION_FRACTION = 0.10
num_validation = max(1, int(round(len(all_document_ids) * VALIDATION_FRACTION)))
num_validation = min(num_validation, len(all_document_ids) - 1)
VAL_DOC_IDS = all_document_ids[:num_validation]
TRAIN_DOC_IDS = all_document_ids[num_validation:]

print(f"\nDocument split: train={len(TRAIN_DOC_IDS):,}, validation={len(VAL_DOC_IDS):,}")
print("Train/validation ID overlap:", len(set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS)))
assert not (set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS))

## 5. Offset-preserving tokenizer and exact character targets

In [ ]:
import re


def _merge_two_offset_items(left_item, right_item, merged_token_id):
    spans = [
        (left_item[1], left_item[2]),
        (right_item[1], right_item[2]),
    ]
    covered = [(start, end) for start, end in spans if start is not None and end is not None]
    if not covered:
        return (merged_token_id, None, None)
    return (
        merged_token_id,
        min(start for start, _ in covered),
        max(end for _, end in covered),
    )


def _encode_word_with_offsets(word, word_start, tokenizer):
    items = []
    char_position = word_start
    for character in word:
        char_start = char_position
        char_end = char_position + len(character)
        for byte_value in character.encode("utf-8"):
            # This mirrors the existing ByteLevelBPE25k._encode_word implementation.
            items.append((4 + byte_value, char_start, char_end))
        char_position = char_end

    # The boundary token has no independent character span. If merged with word bytes,
    # the resulting token retains only the source characters occupied by those bytes.
    items.append((tokenizer.word_boundary_id, None, None))

    while len(items) > 1:
        best_position = None
        best_token_id = None
        for position in range(len(items) - 1):
            pair = (items[position][0], items[position + 1][0])
            merged_id = tokenizer.merge_to_id.get(pair)
            if merged_id is None:
                continue
            if best_token_id is None or merged_id < best_token_id:
                best_token_id = merged_id
                best_position = position
        if best_position is None:
            break
        position = best_position
        merged_item = _merge_two_offset_items(
            items[position], items[position + 1], best_token_id
        )
        items = items[:position] + [merged_item] + items[position + 2:]

    return items


def encode_with_offsets(text, tokenizer, add_bos=False, add_eos=False, verify_ids=True):
    """Return (token_ids, offsets), with offsets as Python-character [start, end) spans or None."""
    token_ids = []
    offsets = []

    if add_bos:
        token_ids.append(tokenizer.bos_token_id)
        offsets.append(None)

    matches = list(re.finditer(r"\S+", text, flags=re.UNICODE))
    regex_words = [match.group(0) for match in matches]
    if regex_words != text.split():
        raise RuntimeError("Whitespace matching differs from tokenizer.encode(text); refusing unsafe offsets.")

    for match in matches:
        word_items = _encode_word_with_offsets(match.group(0), match.start(), tokenizer)
        token_ids.extend(item[0] for item in word_items)
        offsets.extend(
            (item[1], item[2]) if item[1] is not None and item[2] is not None else None
            for item in word_items
        )

    if add_eos:
        token_ids.append(tokenizer.eos_token_id)
        offsets.append(None)

    if verify_ids:
        reference_ids = tokenizer.encode(text, add_bos=add_bos, add_eos=add_eos)
        if token_ids != reference_ids:
            raise AssertionError(
                "Offset-aware tokenization produced different token IDs from the original tokenizer."
            )
    if len(token_ids) != len(offsets):
        raise AssertionError("Token-ID/offset length mismatch.")
    return token_ids, offsets


OFFSET_TEST_STRINGS = [
    "The Supreme Court of India has jurisdiction over constitutional matters.",
    "  Case No. 123/2024\n\nCounsel for the appellant appeared.",
    "Unicode check: न्यायालय and café.",
    "tabs\tand multiple    spaces must preserve original character positions.",
]
for offset_test_text in OFFSET_TEST_STRINGS:
    test_ids, test_offsets = encode_with_offsets(offset_test_text, tokenizer)
    assert test_ids == tokenizer.encode(offset_test_text)
    assert len(test_ids) == len(test_offsets)
    for span in test_offsets:
        if span is not None:
            start, end = span
            assert 0 <= start < end <= len(offset_test_text)
print(f"Offset-aware tokenization checks passed for {len(OFFSET_TEST_STRINGS)} examples.")

In [ ]:
IGNORE_INDEX = -100
BIO_LABELS = ["O"] + [f"B-{label}" for label in LABELS] + [f"I-{label}" for label in LABELS]
BIO_TO_ID = {name: idx for idx, name in enumerate(BIO_LABELS)}
ID_TO_BIO = {idx: name for name, idx in BIO_TO_ID.items()}
O_LABEL_ID = BIO_TO_ID["O"]
print("BIO classes:", len(BIO_LABELS), "| O id:", O_LABEL_ID)

## 6. Exact character-level dataset and windowing

In [ ]:
# Character-level supervision, offset mapping, and dataset construction.
# Gold character offsets are used directly; no tokenizer-boundary approximation is used.

import os
import json
import copy
import bisect
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CHAR_PAD_ID = 0
CHAR_UNK_ID = 1
MAX_CHAR_POSITION_IN_TOKEN = 32

# Build the character vocabulary from TRAIN documents only. Unseen validation/test
# characters map to CHAR_UNK_ID; validation/test text is not used to create it.
_training_chars = set()
for _doc_id in TRAIN_DOC_IDS:
    _training_chars.update(TEXTS_BY_ID[str(_doc_id)])
CHAR_TO_ID = {ch: i + 2 for i, ch in enumerate(sorted(_training_chars))}
CHAR_VOCAB_SIZE = len(CHAR_TO_ID) + 2


def _normalise_entity(entity):
    if isinstance(entity, dict):
        start = entity.get("start", entity.get("start_char", entity.get("start_offset")))
        end = entity.get("end", entity.get("end_char", entity.get("end_offset")))
        label = entity.get("label", entity.get("entity_label", entity.get("type")))
    elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
        start, end, label = entity[:3]
    else:
        raise TypeError(f"Unsupported entity representation: {entity!r}")
    start, end, label = int(start), int(end), str(label)
    if label not in LABELS:
        raise ValueError(f"Unknown entity label {label!r}")
    return start, end, label


def build_exact_character_targets(text, entities, char_start, char_end):
    """Return character BIO target IDs for [char_start, char_end) in the document.

    Fully contained gold entities receive exact B/I tags at original character
    positions. Entities cut by a window boundary are ignored in that window rather
    than taught as truncated entities. Overlapping/nested annotations are masked
    with IGNORE_INDEX because a single BIO sequence cannot represent nested spans faithfully.
    """
    if not (0 <= char_start <= char_end <= len(text)):
        raise ValueError("Invalid character window bounds.")
    length = char_end - char_start
    targets = [O_LABEL_ID] * length
    if length == 0:
        return targets

    parsed = []
    for entity in entities:
        s, e, label = _normalise_entity(entity)
        if not (0 <= s < e <= len(text)):
            raise ValueError(f"Invalid gold span {(s, e, label)} for text length {len(text)}")
        parsed.append((s, e, label))

    # Identify every annotation participating in an overlap/nested span.
    # Such regions are masked with IGNORE_INDEX rather than arbitrarily corrupted.
    conflict_ids = set()
    ordered = sorted(enumerate(parsed), key=lambda item: (item[1][0], item[1][1]))
    for pos, (idx_a, ent_a) in enumerate(ordered):
        s_a, e_a, _ = ent_a
        for idx_b, ent_b in ordered[pos + 1:]:
            s_b, e_b, _ = ent_b
            if s_b >= e_a:
                break
            if s_a < e_b and s_b < e_a:
                conflict_ids.add(idx_a)
                conflict_ids.add(idx_b)

    ignore_positions = set()
    for entity_idx, (s, e, label) in enumerate(parsed):
        overlap_start = max(s, char_start)
        overlap_end = min(e, char_end)
        if overlap_start >= overlap_end:
            continue

        # Only use complete entity examples for supervision in a window.
        is_partial = s < char_start or e > char_end
        if is_partial or entity_idx in conflict_ids:
            ignore_positions.update(range(overlap_start - char_start, overlap_end - char_start))
            continue

        b_id = BIO_TO_ID[f"B-{label}"]
        i_id = BIO_TO_ID[f"I-{label}"]
        local_start = s - char_start
        local_end = e - char_start
        targets[local_start] = b_id
        for p in range(local_start + 1, local_end):
            targets[p] = i_id

    for p in ignore_positions:
        targets[p] = IGNORE_INDEX
    return targets


def _map_characters_to_window_tokens(text, offsets, char_start, char_end):
    """Map each character to its local token index and relative token position.

    Whitespace and any other offset gaps are mapped to the nearest character-bearing
    token. Character targets remain exact; this mapping only supplies contextual
    token features to the character-level head.
    """
    length = char_end - char_start
    char_to_token = [-1] * length
    char_positions = [0] * length

    for token_idx, span in enumerate(offsets):
        if span is None or len(span) < 2 or span[0] is None or span[1] is None:
            continue
        token_start, token_end = int(span[0]), int(span[1])
        if token_start >= token_end:
            continue
        left = max(token_start, char_start)
        right = min(token_end, char_end)
        for absolute_char in range(left, right):
            local_char = absolute_char - char_start
            if 0 <= local_char < length and char_to_token[local_char] == -1:
                char_to_token[local_char] = token_idx
                char_positions[local_char] = min(absolute_char - token_start,
                                                  MAX_CHAR_POSITION_IN_TOKEN - 1)

    covered = [i for i, token_idx in enumerate(char_to_token) if token_idx >= 0]
    if not covered:
        return None, None

    for local_char in range(length):
        if char_to_token[local_char] >= 0:
            continue
        insertion = bisect.bisect_left(covered, local_char)
        left_char = covered[insertion - 1] if insertion > 0 else None
        right_char = covered[insertion] if insertion < len(covered) else None
        if left_char is None:
            source_char = right_char
        elif right_char is None:
            source_char = left_char
        elif local_char - left_char <= right_char - local_char:
            source_char = left_char
        else:
            source_char = right_char
        char_to_token[local_char] = char_to_token[source_char]
        char_positions[local_char] = char_positions[source_char]

    return char_to_token, char_positions


class CharacterBoundaryWindowDataset(Dataset):
    """Sliding token windows with exact character-level BIO supervision.
    Guarantees 100% character coverage across all documents.
    """
    def __init__(self, document_ids, text_by_id, entities_by_id, tokenizer_obj,
                 max_length=512, window_overlap=256):
        if not (0 <= window_overlap < max_length):
            raise ValueError("window_overlap must satisfy 0 <= overlap < max_length")
        self.samples = []
        self.document_ids = [str(x) for x in document_ids]
        step = max_length - window_overlap

        for doc_id in self.document_ids:
            text = text_by_id.get(doc_id, "")
            if not text:
                continue
            entities = entities_by_id.get(doc_id, [])
            token_ids, token_offsets = encode_with_offsets(text, tokenizer_obj)
            token_start = 0
            while token_start < len(token_ids):
                token_end = min(token_start + max_length, len(token_ids))
                window_ids = token_ids[token_start:token_end]
                window_offsets = token_offsets[token_start:token_end]
                valid_offsets = [
                    (int(span[0]), int(span[1]))
                    for span in window_offsets
                    if span is not None and span[0] is not None and span[1] is not None
                    and int(span[0]) < int(span[1])
                ]
                if valid_offsets:
                    char_start = min(s for s, _ in valid_offsets)
                    char_end = max(e for _, e in valid_offsets)

                    # Invariant: extend window 0 to start at char 0 and final window to len(text)
                    if token_start == 0:
                        char_start = 0
                    if token_end == len(token_ids):
                        char_end = len(text)

                    char_to_token, char_positions = _map_characters_to_window_tokens(
                        text, window_offsets, char_start, char_end
                    )
                    if char_to_token is not None:
                        char_ids = [CHAR_TO_ID.get(ch, CHAR_UNK_ID)
                                    for ch in text[char_start:char_end]]
                        char_labels = build_exact_character_targets(
                            text, entities, char_start, char_end
                        )
                        if not (len(char_ids) == len(char_to_token) == len(char_positions)
                                == len(char_labels)):
                            raise AssertionError("Character feature/target lengths differ")
                        if any(t < 0 or t >= len(window_ids) for t in char_to_token):
                            raise AssertionError("Character-to-token mapping is out of bounds")
                        self.samples.append({
                            "input_ids": torch.tensor(window_ids, dtype=torch.long),
                            "char_ids": torch.tensor(char_ids, dtype=torch.long),
                            "char_to_token": torch.tensor(char_to_token, dtype=torch.long),
                            "char_positions": torch.tensor(char_positions, dtype=torch.long),
                            "char_labels": torch.tensor(char_labels, dtype=torch.long),
                            "document_id": doc_id,
                            "token_start": token_start,
                            "char_start": char_start,
                            "char_end": char_end,
                        })
                if token_end == len(token_ids):
                    break
                token_start += step

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def audit_document_character_coverage(texts_by_id, dataset):
    """Audits that every character in each evaluation document is covered by at least one window."""
    coverage_by_doc = {str(doc_id): np.zeros(len(text), dtype=bool) for doc_id, text in texts_by_id.items()}
    for sample in dataset.samples:
        doc_id = sample["document_id"]
        if doc_id in coverage_by_doc:
            s = int(sample["char_start"])
            e = int(sample["char_end"])
            coverage_by_doc[doc_id][s:e] = True

    total_chars = sum(len(cov) for cov in coverage_by_doc.values())
    uncovered_chars = sum(np.sum(~cov) for cov in coverage_by_doc.values())
    coverage_rate = (total_chars - uncovered_chars) / max(total_chars, 1) * 100
    print(f"Document character coverage audit: {coverage_rate:.2f}% ({total_chars - uncovered_chars}/{total_chars} chars covered)")
    if uncovered_chars > 0:
        print(f"Warning: {uncovered_chars} characters were uncovered across {len(coverage_by_doc)} documents.")
    return coverage_by_doc, uncovered_chars == 0


def collate_character_windows(batch, pad_token_id=0):
    """Pads sequences dynamically within batch and returns aligned tensors."""
    if not batch:
        raise ValueError("Cannot collate an empty character-boundary batch")
    max_tokens = max(item["input_ids"].numel() for item in batch)
    max_chars = max(item["char_ids"].numel() for item in batch)
    batch_size = len(batch)

    input_ids = torch.full((batch_size, max_tokens), int(pad_token_id), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_tokens), dtype=torch.float32)
    char_ids = torch.full((batch_size, max_chars), CHAR_PAD_ID, dtype=torch.long)
    char_to_token = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_positions = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_labels = torch.full((batch_size, max_chars), IGNORE_INDEX, dtype=torch.long)
    char_mask = torch.zeros((batch_size, max_chars), dtype=torch.float32)

    document_ids, char_starts, char_ends, char_lengths = [], [], [], []
    for row, item in enumerate(batch):
        token_len = item["input_ids"].numel()
        char_len = item["char_ids"].numel()
        input_ids[row, :token_len] = item["input_ids"]
        attention_mask[row, :token_len] = 1.0
        char_ids[row, :char_len] = item["char_ids"]
        char_to_token[row, :char_len] = item["char_to_token"]
        char_positions[row, :char_len] = item["char_positions"]
        char_labels[row, :char_len] = item["char_labels"]
        char_mask[row, :char_len] = 1.0
        document_ids.append(item["document_id"])
        char_starts.append(int(item["char_start"]))
        char_ends.append(int(item["char_end"]))
        char_lengths.append(int(char_len))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "char_positions": char_positions,
        "char_labels": char_labels,
        "char_mask": char_mask,
        "document_ids": document_ids,
        "char_starts": char_starts,
        "char_ends": char_ends,
        "char_lengths": char_lengths,
    }

print("Character-level supervision, dataset, and collation utilities defined.")
print(f"Character vocabulary size: {CHAR_VOCAB_SIZE} (derived strictly from training split).")

## 7. V2 encoder-decoder architecture (checkpoint-compatible, padding-mask safe)

In [ ]:
# 9. V2 Encoder-Decoder Transformer
# This is our second architecture built from scratch.
# V2 uses:
# - Encoder-decoder Transformer
# - Standard Multi-Head Attention
# - Learned positional embeddings
# - LayerNorm
# - Causal decoder self-attention
# - Decoder cross-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation
#
# We keep the hidden size and attention width close to V1
# so that the comparison mainly reflects the architecture.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# Standard Multi-Head Attention
# ---------------------------------------------------------

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False,
        causal=True,
        key_padding_mask=None
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(x)
        )

        v = self._split_heads(
            self.v_proj(x)
        )

        if past_key_value is not None:

            past_k, past_v = past_key_value

            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # Decoder self-attention is causal.
        # Encoder self-attention is bidirectional.
        #
        # During cached generation, the current token
        # can attend to all previous cached tokens, so
        # no explicit causal mask is needed.
        if causal and past_key_value is None:

            T_q = q.size(-2)
            T_k = k.size(-2)

            causal_mask = torch.tril(
                torch.ones(
                    T_q,
                    T_k,
                    device=x.device,
                    dtype=torch.bool
                )
            )

        else:
            causal_mask = None

        # Correctly mask padded encoder keys. This changes no parameters/state-dict keys.
        attn_mask = causal_mask
        if key_padding_mask is not None:
            key_mask = key_padding_mask[:, None, None, :].bool()
            if attn_mask is None:
                attn_mask = key_mask
            else:
                attn_mask = attn_mask[None, None, :, :] & key_mask

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Cross Attention
# ---------------------------------------------------------

class CrossAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        encoder_output
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(encoder_output)
        )

        v = self._split_heads(
            self.v_proj(encoder_output)
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y


# ---------------------------------------------------------
# Standard GELU Feed Forward Network
# ---------------------------------------------------------

class V2FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x



# ---------------------------------------------------------
# Low-Rank Model Adaptation (LORMA-style residual adapters)
# These bottleneck residual modules are separate from LoRA weight updates.
# ---------------------------------------------------------
class LORMAResidualAdapter(nn.Module):
    def __init__(self, n_embd, rank=8, dropout=0.05):
        super().__init__()
        self.down_proj = nn.Linear(n_embd, rank, bias=False)
        self.up_proj = nn.Linear(rank, n_embd, bias=False)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        # Zero-init the up projection so the pretrained function is unchanged at start.
        nn.init.kaiming_uniform_(self.down_proj.weight, a=math.sqrt(5))
        nn.init.zeros_(self.up_proj.weight)

    def forward(self, x):
        return self.dropout(self.up_proj(self.activation(self.down_proj(x))))


# ---------------------------------------------------------
# Encoder Layer
# ---------------------------------------------------------

class EncoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

        # Added by Phase 2 LORMA. Missing adapter keys are expected when loading
        # the unmodified Phase 1 V2 checkpoint; all original keys must match.
        self.lorma_attention_adapter = LORMAResidualAdapter(
            n_embd=n_embd, rank=LORMA_RANK, dropout=LORMA_DROPOUT
        )
        self.lorma_ffn_adapter = LORMAResidualAdapter(
            n_embd=n_embd, rank=LORMA_RANK, dropout=LORMA_DROPOUT
        )

    def forward(self, x, attention_mask=None):

        # Encoder self-attention is bidirectional.
        normalized_x = self.norm1(x)

        attention_output, _ = self.self_attention(
            normalized_x,
            causal=False,
            key_padding_mask=attention_mask
        )

        x = x + attention_output
        x = x + self.lorma_attention_adapter(x)

        # Feed-forward network.
        x = x + self.ffn(
            self.norm2(x)
        )
        x = x + self.lorma_ffn_adapter(x)

        return x


# ---------------------------------------------------------
# Decoder Layer
# ---------------------------------------------------------

class DecoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.cross_attention = CrossAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        encoder_output,
        past_key_value=None,
        use_cache=False
    ):

        # 1. Decoder causal self-attention.
        attention_output, present = self.self_attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache,
            causal=True
        )

        x = x + attention_output

        # 2. Decoder attends to the encoder output.
        cross_output = self.cross_attention(
            self.norm2(x),
            encoder_output
        )

        x = x + cross_output

        # 3. Feed-forward network.
        x = x + self.ffn(
            self.norm3(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Encoder-Decoder Legal Language Model
# ---------------------------------------------------------

class V2LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_encoder_layers,
        n_decoder_layers,
        n_head,
        source_length,
        target_length,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.source_length = source_length
        self.target_length = target_length

        # Shared token embedding.
        # We also tie this weight to the final LM head.
        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Encoder and decoder use learned positional embeddings.
        self.encoder_position_embedding = nn.Embedding(
            source_length,
            n_embd
        )

        self.decoder_position_embedding = nn.Embedding(
            target_length,
            n_embd
        )

        # Encoder stack.
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_encoder_layers)
        ])

        # Decoder stack.
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_decoder_layers)
        ])

        self.encoder_final_norm = nn.LayerNorm(
            n_embd
        )

        self.decoder_final_norm = nn.LayerNorm(
            n_embd
        )

        # Tied language-model head.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def encode(
        self,
        encoder_input_ids,
        attention_mask=None
    ):

        B, T = encoder_input_ids.shape
        if attention_mask is None:
            attention_mask = encoder_input_ids.ne(PAD_TOKEN_ID)

        positions = torch.arange(
            T,
            device=encoder_input_ids.device
        )

        x = self.token_embedding(
            encoder_input_ids
        )

        x = x + self.encoder_position_embedding(
            positions
        ).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x, attention_mask=attention_mask)

        x = self.encoder_final_norm(x)

        return x

    def decode(
        self,
        decoder_input_ids,
        encoder_output,
        past_key_values=None,
        use_cache=False
    ):

        B, T = decoder_input_ids.shape

        if past_key_values is None:

            position_offset = 0

        else:

            position_offset = (
                past_key_values[0][0].size(-2)
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=decoder_input_ids.device
        )

        x = self.token_embedding(
            decoder_input_ids
        )

        x = x + self.decoder_position_embedding(
            positions
        ).unsqueeze(0)

        presents = [] if use_cache else None

        for layer_index, layer in enumerate(
            self.decoder_layers
        ):

            past = None

            if past_key_values is not None:

                past = past_key_values[
                    layer_index
                ]

            x, present = layer(
                x,
                encoder_output,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:

                presents.append(
                    present
                )

        x = self.decoder_final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

    def forward(
        self,
        encoder_input_ids,
        decoder_input_ids,
        labels=None,
        use_cache=False,
        past_key_values=None,
        encoder_output=None
    ):

        # Encode the source sequence only once.
        if encoder_output is None:

            encoder_output = self.encode(
                encoder_input_ids
            )

        logits, presents = self.decode(
            decoder_input_ids,
            encoder_output,
            past_key_values=past_key_values,
            use_cache=use_cache
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1),
                ignore_index=PAD_TOKEN_ID
            )

        return (
            logits,
            loss,
            presents,
            encoder_output
        )


print("V2 Encoder-Decoder model classes defined.")

## 8. Load the pretrained Phase 1 V2 checkpoint

In [ ]:
if not V2_CHECKPOINT_PATH:
    checkpoint_candidates = []
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                if "v2_model.pt" in files:
                    checkpoint_candidates.append(os.path.join(root, "v2_model.pt"))
    checkpoint_candidates = sorted(set(checkpoint_candidates))
    if len(checkpoint_candidates) == 1:
        V2_CHECKPOINT_PATH = checkpoint_candidates[0]
    elif len(checkpoint_candidates) == 0:
        raise FileNotFoundError(
            "Could not find v2_model.pt. Run the second-architecture Phase 1 notebook and save/download its final v2_model.pt, "
            "attach it as a Kaggle input, then set V2_CHECKPOINT_PATH explicitly."
        )
    else:
        preferred = [p for p in checkpoint_candidates if "final_model" in p]
        if len(preferred) == 1:
            V2_CHECKPOINT_PATH = preferred[0]
        else:
            raise RuntimeError("Multiple v2_model.pt files found. Set V2_CHECKPOINT_PATH explicitly:\n" + "\n".join(checkpoint_candidates))

phase1_checkpoint = torch.load(V2_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if "model_state_dict" not in phase1_checkpoint:
    raise KeyError("Checkpoint must contain 'model_state_dict'. Use the final V2 checkpoint saved by the Phase 1 notebook.")
v2_config = phase1_checkpoint.get("config", {})
if v2_config.get("architecture", "encoder_decoder") != "encoder_decoder":
    raise ValueError(f"Expected V2 encoder-decoder checkpoint; config={v2_config}")

V2_VOCAB_SIZE = int(v2_config.get("vocab_size", tokenizer.vocab_size))
N_EMBD = int(v2_config.get("n_embd", 768))
N_ENCODER_LAYERS = int(v2_config.get("n_encoder_layers", 4))
N_DECODER_LAYERS = int(v2_config.get("n_decoder_layers", 5))
N_HEAD = int(v2_config.get("n_head", 12))
SOURCE_LENGTH = int(v2_config.get("source_length", 512))
TARGET_LENGTH = int(v2_config.get("target_length", 512))
FFN_MULTIPLIER = int(v2_config.get("ffn_multiplier", 4))
DROPOUT = float(v2_config.get("dropout", 0.0))

if V2_VOCAB_SIZE != tokenizer.vocab_size:
    raise ValueError(f"Tokenizer/checkpoint vocabulary mismatch: {tokenizer.vocab_size} vs {V2_VOCAB_SIZE}")
MAX_TOKEN_LENGTH = min(MAX_TOKEN_LENGTH, SOURCE_LENGTH)

v2_backbone = V2LegalLM(
    vocab_size=V2_VOCAB_SIZE, n_embd=N_EMBD,
    n_encoder_layers=N_ENCODER_LAYERS, n_decoder_layers=N_DECODER_LAYERS,
    n_head=N_HEAD, source_length=SOURCE_LENGTH, target_length=TARGET_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER, dropout=DROPOUT
)
# V2LegalLM's generic initializer visits every Linear; re-zero the new adapter up-projections
# so each adapter begins as an exact zero residual and does not disturb the Phase 1 function.
for encoder_layer in v2_backbone.encoder_layers:
    nn.init.zeros_(encoder_layer.lorma_attention_adapter.up_proj.weight)
    nn.init.zeros_(encoder_layer.lorma_ffn_adapter.up_proj.weight)

load_result = v2_backbone.load_state_dict(phase1_checkpoint["model_state_dict"], strict=False)
# The only missing keys allowed are the newly initialized Phase 2 LORMA residual adapters.
missing_keys = list(load_result.missing_keys)
unexpected_keys = list(load_result.unexpected_keys)
if unexpected_keys:
    raise RuntimeError(f"Unexpected checkpoint keys; architecture/checkpoint mismatch: {unexpected_keys[:20]}")
if any(".lorma_attention_adapter." not in k and ".lorma_ffn_adapter." not in k for k in missing_keys):
    raise RuntimeError(f"Missing non-LORMA pretrained keys; refusing to continue: {missing_keys[:20]}")
expected_adapter_key_count = N_ENCODER_LAYERS * 2 * 2
if len(missing_keys) != expected_adapter_key_count:
    raise RuntimeError(f"Expected {expected_adapter_key_count} fresh LORMA adapter parameter tensors, got {len(missing_keys)}.")
v2_backbone = v2_backbone.to(DEVICE)
print("Loaded V2 checkpoint:", V2_CHECKPOINT_PATH)
print("Original V2 weights loaded strictly except for newly added LORMA adapters.")
print("Expected fresh adapter tensors:", len(missing_keys))
print("V2 config:", {k: v2_config.get(k) for k in ["n_embd","n_encoder_layers","n_decoder_layers","n_head","source_length","target_length"]})
print("Strict checkpoint load passed:", load_result)

## 9. LORMA low-rank residual adapters

In [ ]:
# LORMA residual adapters are already part of the patched EncoderLayer definition.
# The Phase 1 checkpoint was loaded before training, with only the new adapter tensors missing.
# Freeze every pretrained V2 parameter and enable only the LORMA residual adapters.
for name, parameter in v2_backbone.named_parameters():
    parameter.requires_grad = (".lorma_attention_adapter." in name or ".lorma_ffn_adapter." in name)

LORMA_TARGETS = []
for layer_idx, layer in enumerate(v2_backbone.encoder_layers):
    LORMA_TARGETS.extend([
        f"encoder_layers.{layer_idx}.lorma_attention_adapter",
        f"encoder_layers.{layer_idx}.lorma_ffn_adapter",
    ])

print("Inserted residual LORMA adapters:", len(LORMA_TARGETS))
print("\n".join(LORMA_TARGETS))
print("Trainable LORMA adapter parameters:",
      sum(p.numel() for p in v2_backbone.parameters() if p.requires_grad))


## 10. Character-level NER head and LORMA model

In [ ]:
class CharacterBoundaryHead(nn.Module):
    """Emit BIO logits for each raw-text character using contextual token features.
    Substantive modelling improvements:
    - Multi-scale dilated convolutions (local sub-word context + full-word dilated context)
    - Channel LayerNorm for training stability
    - Residual connections across convolutional stages
    """
    def __init__(self, hidden_dim, char_vocab_size, num_labels, char_dim=32,
                 token_feature_dim=128, relative_position_dim=16, dropout=0.1):
        super().__init__()
        self.token_projection = nn.Linear(hidden_dim, token_feature_dim)
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_dim, padding_idx=CHAR_PAD_ID
        )
        self.relative_position_embedding = nn.Embedding(
            MAX_CHAR_POSITION_IN_TOKEN, relative_position_dim
        )
        joined_dim = token_feature_dim + char_dim + relative_position_dim
        self.input_projection = nn.Sequential(
            nn.Linear(joined_dim, token_feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.norm0 = nn.LayerNorm(token_feature_dim)

        # Conv1: receptive field 5 (local char context: sub-word/affixes)
        self.conv1 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=2)
        self.norm1 = nn.LayerNorm(token_feature_dim)

        # Conv2: dilated conv with dilation 2 (receptive field spans ~13 chars: full word context)
        self.conv2 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=4, dilation=2)
        self.norm2 = nn.LayerNorm(token_feature_dim)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(token_feature_dim, num_labels)

    def forward(self, token_hidden_states, char_ids, char_to_token,
                char_positions, char_mask):
        batch_size, char_length = char_ids.shape
        safe_map = char_to_token.clamp(min=0, max=token_hidden_states.shape[1] - 1)
        gather_index = safe_map.unsqueeze(-1).expand(-1, -1, token_hidden_states.shape[-1])
        char_context = token_hidden_states.gather(dim=1, index=gather_index)

        features = torch.cat([
            self.token_projection(char_context),
            self.char_embedding(char_ids),
            self.relative_position_embedding(
                char_positions.clamp(min=0, max=MAX_CHAR_POSITION_IN_TOKEN - 1)
            ),
        ], dim=-1)
        x = self.input_projection(features)
        mask = char_mask.unsqueeze(-1)
        x = self.norm0(x) * mask

        # Conv1 with residual
        conv1_out = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        h1 = self.norm1(F.gelu(conv1_out) + x) * mask
        h1 = self.dropout(h1)

        # Conv2 with residual
        conv2_out = self.conv2(h1.transpose(1, 2)).transpose(1, 2)
        h2 = self.norm2(F.gelu(conv2_out) + h1) * mask
        h2 = self.dropout(h2)

        return self.classifier(h2)



class V2LORMACharacterNERModel(nn.Module):
    def __init__(self, backbone, char_vocab_size, num_labels):
        super().__init__()
        self.backbone = backbone
        self.char_head = CharacterBoundaryHead(
            hidden_dim=N_EMBD,
            char_vocab_size=char_vocab_size,
            num_labels=num_labels,
            dropout=0.1
        )

    def forward(self, input_ids, attention_mask, char_ids, char_to_token,
                char_positions, char_mask):
        token_hidden = self.backbone.encode(input_ids, attention_mask=attention_mask.bool())
        return self.char_head(token_hidden, char_ids, char_to_token, char_positions, char_mask)

lorma_model = V2LORMACharacterNERModel(
    backbone=v2_backbone,
    char_vocab_size=CHAR_VOCAB_SIZE,
    num_labels=len(BIO_LABELS)
).to(DEVICE)

# The pretrained backbone is frozen; adapter matrices plus character head are trainable.
for p in lorma_model.char_head.parameters():
    p.requires_grad = True
trainable_params = [(n,p) for n,p in lorma_model.named_parameters() if p.requires_grad]
total_params = sum(p.numel() for p in lorma_model.parameters())
trainable_count = sum(p.numel() for _,p in trainable_params)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_count:,} ({100*trainable_count/max(total_params,1):.3f}%)")
assert trainable_count > 0
assert all(("lorma_attention_adapter" in n or "lorma_ffn_adapter" in n or "char_head" in n) for n,p in trainable_params), "Unexpected trainable base parameter detected."

## LORMA pre-flight smoke test

This runs a tiny forward/backward pass through the loaded V2 encoder, adapters, and character head. It checks that the real checkpoint can produce finite logits and that gradients reach adapters/head while frozen base weights remain frozen. It does **not** start training.


In [ ]:
# Lightweight preflight against the real loaded V2 checkpoint; no optimizer step and no training.
lorma_model.eval()
_smoke_len = min(16, MAX_TOKEN_LENGTH)
_smoke_chars = 32
_smoke_ids = torch.randint(0, tokenizer.vocab_size, (_smoke_batch := 2, _smoke_len), device=DEVICE)
_smoke_attention = torch.ones_like(_smoke_ids)
_smoke_char_ids = torch.randint(2, max(3, CHAR_VOCAB_SIZE), (_smoke_batch, _smoke_chars), device=DEVICE)
_smoke_char_to_token = torch.randint(0, _smoke_len, (_smoke_batch, _smoke_chars), device=DEVICE)
_smoke_char_positions = torch.randint(0, MAX_CHAR_POSITION_IN_TOKEN, (_smoke_batch, _smoke_chars), device=DEVICE)
_smoke_char_mask = torch.ones((_smoke_batch, _smoke_chars), device=DEVICE)
_smoke_labels = torch.randint(0, len(BIO_LABELS), (_smoke_batch, _smoke_chars), device=DEVICE)

lorma_model.zero_grad(set_to_none=True)
with torch.cuda.amp.autocast(enabled=USE_AMP):
    _smoke_logits = lorma_model(
        _smoke_ids, _smoke_attention, _smoke_char_ids,
        _smoke_char_to_token, _smoke_char_positions, _smoke_char_mask
    )
    _smoke_loss = F.cross_entropy(_smoke_logits.reshape(-1, len(BIO_LABELS)), _smoke_labels.reshape(-1))
assert torch.isfinite(_smoke_logits).all().item(), "Non-finite LORMA smoke-test logits."
assert torch.isfinite(_smoke_loss).item(), "Non-finite LORMA smoke-test loss."
_smoke_loss.backward()
_adapter_with_grad = [
    name for name, param in lorma_model.named_parameters()
    if param.requires_grad and ("lorma_attention_adapter" in name or "lorma_ffn_adapter" in name) and param.grad is not None
]
_head_with_grad = any(
    param.grad is not None and torch.isfinite(param.grad).all().item()
    for param in lorma_model.char_head.parameters()
)
assert _adapter_with_grad, "No LORMA adapter received gradients."
assert _head_with_grad, "Character head did not receive finite gradients."
assert all(
    param.grad is None for name, param in lorma_model.named_parameters()
    if not param.requires_grad
), "A frozen base parameter unexpectedly received a gradient."
lorma_model.zero_grad(set_to_none=True)
print("LORMA pre-flight passed. Loss:", round(float(_smoke_loss.detach().cpu()), 4))
print("Adapter matrices receiving gradients:", len(_adapter_with_grad))
print("Frozen base parameters stayed frozen.")


## 11. Training, exact-span evaluation, and checkpoint selection

In [ ]:
def strict_span_metrics(gold_by_id, predicted_by_id, labels, document_ids):
    per_label = {}
    f1s = []
    for label in labels:
        gold_set, pred_set = set(), set()
        for doc_id in document_ids:
            for ent in gold_by_id.get(str(doc_id), []):
                if str(ent["label"]) == label:
                    gold_set.add((str(doc_id), int(ent["start"]), int(ent["end"]), label))
            for ent in predicted_by_id.get(str(doc_id), []):
                if str(ent[2]) == label:
                    pred_set.add((str(doc_id), int(ent[0]), int(ent[1]), label))
        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        per_label[label] = {"tp":tp,"fp":fp,"fn":fn,"precision":p,"recall":r,"f1":f1}
        f1s.append(f1)
    macro_p = sum(v["precision"] for v in per_label.values()) / max(1,len(labels))
    macro_r = sum(v["recall"] for v in per_label.values()) / max(1,len(labels))
    return {
        "strict_macro_f1": sum(f1s)/max(1,len(f1s)),
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "total_true_positives": sum(v["tp"] for v in per_label.values()),
        "total_false_positives": sum(v["fp"] for v in per_label.values()),
        "total_false_negatives": sum(v["fn"] for v in per_label.values()),
        "per_label": per_label,
    }


def _character_model_inputs(batch, device):
    return {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
        "char_ids": batch["char_ids"].to(device),
        "char_to_token": batch["char_to_token"].to(device),
        "char_positions": batch["char_positions"].to(device),
        "char_mask": batch["char_mask"].to(device),
    }


def collect_document_character_logits(model, dataset, texts_by_id, batch_size=2):
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID)
    )
    model.eval()
    accumulated, accumulated_weight = {}, {}
    with torch.inference_mode():
        for batch in loader:
            inputs = _character_model_inputs(batch, DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**inputs)
            logits_cpu = logits.float().cpu().numpy()
            for row, doc_id in enumerate(batch["document_ids"]):
                text = texts_by_id[str(doc_id)]
                if str(doc_id) not in accumulated:
                    accumulated[str(doc_id)] = np.zeros((len(text), len(BIO_LABELS)), dtype=np.float32)
                    accumulated_weight[str(doc_id)] = np.zeros(len(text), dtype=np.float32)
                start = int(batch["char_starts"][row])
                length = int(batch["char_lengths"][row])
                if length <= 0:
                    continue
                end = min(start + length, len(text))
                length = end - start
                positions = np.arange(length)
                edge = np.minimum(positions, length - 1 - positions)
                weights = 1.0 + np.minimum(edge, 64).astype(np.float32) / 64.0
                accumulated[str(doc_id)][start:end] += logits_cpu[row, :length, :] * weights[:, None]
                accumulated_weight[str(doc_id)][start:end] += weights
    result = {}
    for doc_id, text in texts_by_id.items():
        doc_id = str(doc_id)
        if len(text) == 0:
            result[doc_id] = np.zeros((0, len(BIO_LABELS)), dtype=np.float32)
        elif doc_id not in accumulated:
            fallback = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
            fallback[:, O_LABEL_ID] = 0.0
            result[doc_id] = fallback
        else:
            weights = accumulated_weight[doc_id]
            covered = weights > 0
            out = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
            out[:, O_LABEL_ID] = 0.0
            out[covered] = accumulated[doc_id][covered] / weights[covered, None]
            result[doc_id] = out
    return result


def decode_character_bio_logits(logits, thresholds=None, min_entity_chars=2):
    arr = np.asarray(logits, dtype=np.float32).copy()
    if thresholds:
        for label, bias in thresholds.items():
            arr[:, BIO_TO_ID[f"B-{label}"]] -= float(bias)
            arr[:, BIO_TO_ID[f"I-{label}"]] -= float(bias)
    tag_ids = arr.argmax(axis=-1).tolist()
    spans = []
    active_label, active_start = None, None

    def flush(end):
        nonlocal active_label, active_start
        if active_label is not None and active_start is not None and end - active_start >= min_entity_chars:
            spans.append([int(active_start), int(end), str(active_label)])
        active_label, active_start = None, None

    for pos, tag_id in enumerate(tag_ids):
        tag = ID_TO_BIO[int(tag_id)]
        if tag.startswith("B-"):
            flush(pos)
            active_label, active_start = tag[2:], pos
        elif tag.startswith("I-"):
            label = tag[2:]
            if active_label != label:
                flush(pos)
                # Strict BIO: orphan I tags do not start new entities.
            # same label means continuation
        else:
            flush(pos)
    flush(len(tag_ids))
    return spans


def score_document_logits(logits_by_doc, texts_by_id, gold_by_id, thresholds=None):
    preds = {str(doc_id): decode_character_bio_logits(logits, thresholds=thresholds)
             for doc_id, logits in logits_by_doc.items()}
    ids = [str(x) for x in texts_by_id]
    for doc_id in ids:
        preds.setdefault(doc_id, [])
    gold = {str(k): v for k,v in gold_by_id.items()}
    metrics = strict_span_metrics(gold, preds, LABELS, ids)
    metrics["predictions_by_doc"] = preds
    return metrics


def _char_loss_weights(dataset, device):
    counts = torch.zeros(len(BIO_LABELS), dtype=torch.float64)
    for sample in dataset.samples:
        target = sample["char_labels"]
        target = target[target != IGNORE_INDEX]
        if target.numel():
            counts += torch.bincount(target, minlength=len(BIO_LABELS)).double()
    counts = counts.float().clamp_min(1.0)
    o_count = counts[O_LABEL_ID]
    # Conservative smoothing: do not crush O or give rare tags unbounded weights.
    weights = (o_count / counts).pow(0.15).clamp(1.0, 2.0)
    weights[O_LABEL_ID] = 1.0
    print("Loss weights: O=1.000; entity weights min/max =", round(float(weights.min()),3), round(float(weights.max()),3))
    return weights.to(device)


class WeightedFocalLoss(nn.Module):
    def __init__(self, weights, gamma=1.0, ignore_index=-100):
        super().__init__()
        self.register_buffer("weights", weights.detach().float().clone())
        self.gamma = float(gamma)
        self.ignore_index = int(ignore_index)
    def forward(self, logits, targets):
        logits = logits.reshape(-1, logits.shape[-1])
        targets = targets.reshape(-1).long()
        valid = targets.ne(self.ignore_index)
        if not valid.any():
            return logits.sum() * 0.0
        logp = F.log_softmax(logits[valid], dim=-1)
        y = targets[valid]
        log_pt = logp.gather(1, y[:,None]).squeeze(1)
        alpha = self.weights[y]
        loss = -alpha * (1.0 - log_pt.exp()).clamp(0,1).pow(self.gamma) * log_pt
        return loss.sum() / alpha.sum().clamp_min(1e-8)


def evaluate_lorma(model, dataset, texts, gold, batch_size=2):
    logits = collect_document_character_logits(model, dataset, texts, batch_size=batch_size)
    metrics = score_document_logits(logits, texts, gold, thresholds=None)
    metrics["document_logits"] = logits
    return metrics


train_texts = {str(k): TEXTS_BY_ID[str(k)] for k in TRAIN_DOC_IDS}
val_texts = {str(k): TEXTS_BY_ID[str(k)] for k in VAL_DOC_IDS}
train_gold = {str(k): ANNOTATIONS_BY_ID[str(k)] for k in TRAIN_DOC_IDS}
val_gold = {str(k): ANNOTATIONS_BY_ID[str(k)] for k in VAL_DOC_IDS}

train_dataset = CharacterBoundaryWindowDataset(
    TRAIN_DOC_IDS, train_texts, train_gold, tokenizer,
    max_length=MAX_TOKEN_LENGTH, window_overlap=WINDOW_OVERLAP
)
val_dataset = CharacterBoundaryWindowDataset(
    VAL_DOC_IDS, val_texts, val_gold, tokenizer,
    max_length=MAX_TOKEN_LENGTH, window_overlap=WINDOW_OVERLAP
)
assert len(train_dataset) and len(val_dataset), "Train or validation windows are empty."
audit_document_character_coverage(val_texts, val_dataset)
print("Windows:", len(train_dataset), "train /", len(val_dataset), "validation")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID)
)
criterion = WeightedFocalLoss(_char_loss_weights(train_dataset, DEVICE), gamma=1.0, ignore_index=IGNORE_INDEX).to(DEVICE)

trainable = [p for p in lorma_model.parameters() if p.requires_grad]
adapter_params = [p for n,p in lorma_model.named_parameters() if p.requires_grad and ("lorma_attention_adapter" in n or "lorma_ffn_adapter" in n)]
head_params = [p for n,p in lorma_model.named_parameters() if p.requires_grad and "char_head" in n]
optimizer = torch.optim.AdamW(
    [{"params": adapter_params, "lr": LORMA_LR},
     {"params": head_params, "lr": CHAR_HEAD_LR}],
    weight_decay=WEIGHT_DECAY
)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = max(1, steps_per_epoch * EPOCHS)
warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))
def lr_lambda(step):
    if step < warmup_steps:
        return max(0.05, (step + 1) / warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

BEST_LORMA_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "best_v2_lorma_character_boundary.pt")
history = []
best_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    lorma_model.train()
    running_loss, micro_steps = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    epoch_start = time.time()
    for step, batch in enumerate(train_loader, start=1):
        inputs = _character_model_inputs(batch, DEVICE)
        labels = batch["char_labels"].to(DEVICE)
        group_start_step = ((step - 1) // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_size = min(GRAD_ACCUM_STEPS, len(train_loader) - group_start_step)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = lorma_model(**inputs)
            unscaled_loss = criterion(logits, labels)
            loss = unscaled_loss / group_size
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss at epoch {epoch}, step {step}.")
        scaler.scale(loss).backward()
        running_loss += float(unscaled_loss.item())
        micro_steps += 1
        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    val_metrics = evaluate_lorma(lorma_model, val_dataset, val_texts, val_gold, batch_size=BATCH_SIZE)
    row = {
        "epoch": epoch,
        "train_loss": running_loss / max(1, micro_steps),
        "val_strict_macro_f1": float(val_metrics["strict_macro_f1"]),
        "val_macro_precision": float(val_metrics["macro_precision"]),
        "val_macro_recall": float(val_metrics["macro_recall"]),
        "tp": int(val_metrics["total_true_positives"]),
        "fp": int(val_metrics["total_false_positives"]),
        "fn": int(val_metrics["total_false_negatives"]),
        "seconds": time.time() - epoch_start,
    }
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={row['train_loss']:.4f} | "
          f"strict_macro_f1={row['val_strict_macro_f1']:.4f} | P={row['val_macro_precision']:.4f} "
          f"R={row['val_macro_recall']:.4f} | TP={row['tp']} FP={row['fp']} FN={row['fn']} "
          f"| {row['seconds']:.1f}s")
    for label in LABELS:
        m = val_metrics["per_label"][label]
        print(f"  {label:10s} F1={m['f1']:.3f} P={m['precision']:.3f} R={m['recall']:.3f} TP={m['tp']} FP={m['fp']} FN={m['fn']}")

    if row["val_strict_macro_f1"] > best_f1 + 1e-6:
        best_f1 = row["val_strict_macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": lorma_model.state_dict(),
            "epoch": epoch,
            "val_f1": best_f1,
            "val_metrics": {k:v for k,v in val_metrics.items() if k not in ("predictions_by_doc","document_logits")},
            "history": history,
            "config": {
                "architecture": "V2 encoder + LORMA adapters + character boundary head",
                "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT,
                "labels": LABELS, "bio_labels": BIO_LABELS,
                "char_to_id": CHAR_TO_ID, "char_vocab_size": CHAR_VOCAB_SIZE,
                "max_token_length": MAX_TOKEN_LENGTH, "window_overlap": WINDOW_OVERLAP,
                "source_checkpoint": V2_CHECKPOINT_PATH,
            }
        }, BEST_LORMA_CHECKPOINT)
        print("  Saved best checkpoint:", BEST_LORMA_CHECKPOINT)
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {epoch} epochs: no strict-F1 improvement for {PATIENCE} epochs.")
        break

# Restore the best epoch checkpoint before calibration/inference.
best_state = torch.load(BEST_LORMA_CHECKPOINT, map_location=DEVICE, weights_only=False)
lorma_model.load_state_dict(best_state["model_state_dict"], strict=True)
print(f"Restored best LORMA checkpoint: epoch={best_state['epoch']} val_f1={best_state['val_f1']:.4f}")
with open(os.path.join(OUTPUT_DIR, "lorma_training_history.json"), "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

## 12. Validation-only confidence calibration

In [ ]:
# Greedy coordinate search over conservative per-label biases.
# This uses validation labels only; do not interpret calibrated validation F1 as an unbiased estimate.
calibration_logits = collect_document_character_logits(lorma_model, val_dataset, val_texts, batch_size=BATCH_SIZE)
raw_val_metrics = score_document_logits(calibration_logits, val_texts, val_gold, thresholds=None)
thresholds = {label: 0.0 for label in LABELS}
best_metrics = raw_val_metrics
candidates = [0.0, 0.5, 1.0, 1.5, 2.0]
print(f"Raw validation: F1={raw_val_metrics['strict_macro_f1']:.4f}, P={raw_val_metrics['macro_precision']:.4f}, R={raw_val_metrics['macro_recall']:.4f}, FP={raw_val_metrics['total_false_positives']}")

for pass_idx in range(2):
    changed = False
    for label in LABELS:
        current_bias = thresholds[label]
        current_metrics = best_metrics
        current_score = current_metrics["strict_macro_f1"]
        current_fp = current_metrics["per_label"][label]["fp"]
        current_tp = current_metrics["per_label"][label]["tp"]
        selected_bias = current_bias
        for candidate in candidates:
            trial = dict(thresholds)
            trial[label] = candidate
            trial_metrics = score_document_logits(calibration_logits, val_texts, val_gold, thresholds=trial)
            score = trial_metrics["strict_macro_f1"]
            tp = trial_metrics["per_label"][label]["tp"]
            fp = trial_metrics["per_label"][label]["fp"]
            if score > current_score + 1e-7 or (
                abs(score-current_score) <= 1e-7 and tp >= current_tp and fp < current_fp
            ):
                current_score, current_metrics = score, trial_metrics
                current_tp, current_fp, selected_bias = tp, fp, candidate
        if selected_bias != current_bias:
            changed = True
        thresholds[label] = selected_bias
        best_metrics = current_metrics
        print(f"Pass {pass_idx+1} | {label:10s} bias={selected_bias:.1f} | macro F1={best_metrics['strict_macro_f1']:.4f}")
    if not changed:
        break

if best_metrics["strict_macro_f1"] + 1e-8 < raw_val_metrics["strict_macro_f1"]:
    thresholds = {label: 0.0 for label in LABELS}
    best_metrics = raw_val_metrics

print("\nCalibration summary")
print(f"Raw F1={raw_val_metrics['strict_macro_f1']:.4f}; calibrated F1={best_metrics['strict_macro_f1']:.4f}")
print("Thresholds:", thresholds)
print("Per-label calibrated validation metrics:")
for label in LABELS:
    m = best_metrics["per_label"][label]
    print(f"  {label:10s} F1={m['f1']:.3f} P={m['precision']:.3f} R={m['recall']:.3f} TP={m['tp']} FP={m['fp']} FN={m['fn']}")
with open(os.path.join(OUTPUT_DIR, "lorma_calibration.json"), "w", encoding="utf-8") as f:
    json.dump({"thresholds": thresholds, "raw_f1": raw_val_metrics["strict_macro_f1"],
               "calibrated_val_f1": best_metrics["strict_macro_f1"]}, f, indent=2)

## 13. Test inference and submission ZIP (no automatic submission)

In [ ]:
test_texts = {str(k): v for k, v in TEST_TEXTS.items()}
test_dataset = CharacterBoundaryWindowDataset(
    document_ids=list(test_texts.keys()),
    text_by_id=test_texts,
    entities_by_id={},
    tokenizer_obj=tokenizer,
    max_length=MAX_TOKEN_LENGTH,
    window_overlap=WINDOW_OVERLAP,
)
audit_document_character_coverage(test_texts, test_dataset)
print(f"Test documents: {len(test_texts):,}; test windows: {len(test_dataset):,}")

test_logits = collect_document_character_logits(lorma_model, test_dataset, test_texts, batch_size=BATCH_SIZE)
test_predictions = {
    str(doc_id): decode_character_bio_logits(logits, thresholds=thresholds)
    for doc_id, logits in test_logits.items()
}
for doc_id in test_texts:
    test_predictions.setdefault(str(doc_id), [])

# Validate document coverage, labels, offsets, and duplicates before packaging.
expected_ids, actual_ids = set(test_texts), set(test_predictions)
if expected_ids != actual_ids:
    raise ValueError(f"Test ID mismatch: missing={len(expected_ids-actual_ids)}, extra={len(actual_ids-expected_ids)}")
for doc_id, entities in test_predictions.items():
    seen = set()
    for start, end, label in entities:
        if label not in LABELS or not (0 <= start < end <= len(test_texts[doc_id])):
            raise ValueError(f"Invalid prediction for {doc_id}: {(start,end,label)}")
        triple = (start,end,label)
        if triple in seen:
            raise ValueError(f"Duplicate prediction for {doc_id}: {triple}")
        seen.add(triple)

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission_v2_lorma_character_boundary.zip")
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    packaged = json.loads(zf.read("prediction.json").decode("utf-8"))
assert set(packaged) == expected_ids
print("Prediction JSON:", PREDICTION_PATH)
print("Submission ZIP:", SUBMISSION_ZIP)
print("Documents:", len(packaged), "| Total spans:", sum(len(v) for v in packaged.values()))
print("ZIP validation passed. Nothing has been submitted.")

## 14. Save experiment manifest and final notes

In [ ]:
manifest = {
    "experiment_id": "EXP-2-V2-LORMA-CharacterBoundary",
    "source_checkpoint": V2_CHECKPOINT_PATH,
    "tokenizer_path": BPE_PATH,
    "architecture": "V2 encoder-decoder pretrained model; encoder used for NER",
    "adaptation": "LORMA low-rank bottleneck residual adapters after encoder attention and feed-forward sublayers",
    "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT,
    "epochs_requested": EPOCHS, "best_epoch": int(best_state["epoch"]),
    "best_raw_validation_strict_macro_f1": float(best_state["val_f1"]),
    "raw_validation_metrics": {
        "strict_macro_f1": float(raw_val_metrics["strict_macro_f1"]),
        "macro_precision": float(raw_val_metrics["macro_precision"]),
        "macro_recall": float(raw_val_metrics["macro_recall"]),
        "tp": int(raw_val_metrics["total_true_positives"]),
        "fp": int(raw_val_metrics["total_false_positives"]),
        "fn": int(raw_val_metrics["total_false_negatives"]),
    },
    "calibrated_validation_strict_macro_f1": float(best_metrics["strict_macro_f1"]),
    "thresholds": thresholds,
    "trainable_parameters": int(trainable_count),
    "total_parameters": int(total_params),
    "submission_zip": SUBMISSION_ZIP,
    "submission_sha256": hashlib.sha256(open(SUBMISSION_ZIP, "rb").read()).hexdigest(),
}
with open(os.path.join(OUTPUT_DIR, "experiment_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))